# NOTEBOOK : 08_stacking.ipynb
# ROLE     : Build a Level-2 Stacking ensemble from the top 2 validation models (no retraining of Level-1 models).
# INPUT    : ../../data/processed/val.csv, ../artifacts/best_two_models.json, ../artifacts/models/*.joblib
# OUTPUT   : ../artifacts/models/stacking_model.joblib

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score

PROCESSED_DIR = Path("../../data/processed")
ARTIFACTS_DIR = Path("../artifacts")
MODELS_DIR = ARTIFACTS_DIR / "models"

val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
with open(ARTIFACTS_DIR / "features_list.json", "r", encoding="utf-8") as f:
    FEATURE_COLS = json.load(f)

with open(ARTIFACTS_DIR / "best_two_models.json", "r", encoding="utf-8") as f:
    best_two_info = json.load(f)["top_two_models"]

scaler = joblib.load(MODELS_DIR / "match_scaler.joblib")

X_val = val_df[FEATURE_COLS]
X_val_scaled = scaler.transform(X_val)
y_val = val_df["target"].values  # 0=A, 1=D, 2=H

m1_info, m2_info = best_two_info[0], best_two_info[1]
m1_name, m1_scaled = m1_info["model"], m1_info["uses_scaled"]
m2_name, m2_scaled = m2_info["model"], m2_info["uses_scaled"]

model_1 = joblib.load(MODELS_DIR / f"{m1_name}.joblib")
model_2 = joblib.load(MODELS_DIR / f"{m2_name}.joblib")

print(f"Level-1 Model 1: {m1_name} (uses_scaled={m1_scaled})")
print(f"Level-1 Model 2: {m2_name} (uses_scaled={m2_scaled})")

Level-1 Model 1: random_forest (uses_scaled=False)
Level-1 Model 2: logistic_regression (uses_scaled=True)


In [2]:
# Level 1: Predict (P_A, P_D, P_H) on val.csv using the already-trained base models
X_val_1 = X_val_scaled if m1_scaled else X_val
X_val_2 = X_val_scaled if m2_scaled else X_val

probs_1 = model_1.predict_proba(X_val_1)  # shape (N, 3)
probs_2 = model_2.predict_proba(X_val_2)  # shape (N, 3)

# Concatenate into a 6-column meta-feature matrix
X_meta_val = np.hstack([probs_1, probs_2])
print(f"Meta-feature matrix shape: {X_meta_val.shape}")

# Level 2: Train a simple Multinomial Logistic Regression meta-model
meta_model = LogisticRegression(max_iter=1000, random_state=42)
meta_model.fit(X_meta_val, y_val)

# Sanity check on validation set (in-sample for meta-model; true test is in Notebook 09)
meta_probs = meta_model.predict_proba(X_meta_val)
meta_preds = meta_model.predict(X_meta_val)
print(f"Meta-model (val in-sample) Log Loss : {log_loss(y_val, meta_probs, labels=[0, 1, 2]):.4f}")
print(f"Meta-model (val in-sample) Accuracy : {accuracy_score(y_val, meta_preds):.4f}")

Meta-feature matrix shape: (190, 6)
Meta-model (val in-sample) Log Loss : 0.9961
Meta-model (val in-sample) Accuracy : 0.5105


In [3]:
# IMPORTANT: model_1 and model_2 are NOT retrained on train+val to prevent leakage into meta_model.
stacking_artifact = {
    "model_1_name": m1_name,
    "model_2_name": m2_name,
    "model_1_uses_scaled": m1_scaled,
    "model_2_uses_scaled": m2_scaled,
    "model_1": model_1,
    "model_2": model_2,
    "meta_model": meta_model,
}

stacking_path = MODELS_DIR / "stacking_model.joblib"
joblib.dump(stacking_artifact, stacking_path)
print(f"[SAVED] Stacking ensemble -> {stacking_path}")

[SAVED] Stacking ensemble -> ..\artifacts\models\stacking_model.joblib
